# End-to-End API Walkthrough

This generated notebook is the recipe companion for
`examples/api_end_to_end_demo.py`.

Run the full recipe: build raw and processed datasets, merge them into one working dataset, build a report, save YAML, and render both PDF and PNG outputs.

What you will practice in this walkthrough:

- Connect ingestion, processing, merge, layout, rendering, and serialization in one workflow.
- Keep raw and processed channels visible together inside a single working dataset.
- Produce both final artifacts and lighter preview artifacts from the same report definition.

Prerequisites:

- `pip install "wellplot[notebook,pandas]"`
- run the notebook from a checkout of this repository so the `examples/` files and sample data are available

Runtime model:

- import `wellplot` from the active installed environment
- use the repository checkout for the example files, helper modules, and sample data


In [ ]:
import sys
from pathlib import Path

try:
    import wellplot
except ImportError as exc:
    raise RuntimeError(
        "Install the published 'wellplot' package in the active "
        "environment before running this notebook."
    ) from exc

# Walk upward from the current working directory until we find the
# repository checkout that holds the example sources and sample data.
cwd = Path.cwd().resolve()
REPO_ROOT = next((path for path in (cwd, *cwd.parents) if (path / "examples").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError(
        "Run this notebook from a checkout of the wellplot repository "
        "so the example files and sample data are available."
    )

EXAMPLES_DIR = REPO_ROOT / "examples"
WORKSPACE_DIR = REPO_ROOT / "workspace"
WORKSPACE_RENDERS = WORKSPACE_DIR / "renders"
WORKSPACE_RENDERS.mkdir(parents=True, exist_ok=True)

examples_path = str(EXAMPLES_DIR)
if examples_path not in sys.path:
    sys.path.insert(0, examples_path)

print("wellplot version:", wellplot.__version__)
print("Examples root:", EXAMPLES_DIR)
print("Render output:", WORKSPACE_RENDERS)

In [ ]:
# Display the source directly in the notebook so the recipe is easy to
# read and copy from without opening another file.
from IPython.display import Code, display

source_path = EXAMPLES_DIR / "api_end_to_end_demo.py"
display(Code(source_path.read_text(), language="python"))

In [ ]:
# Import the example module and build the merged working dataset used by the
# end-to-end recipe.
import api_end_to_end_demo as demo

dataset = demo.build_working_dataset()
print("Channels:", sorted(dataset.channels))
print("Depth range:", dataset.depth_range("ft"))

In [ ]:
# Build the report and save a normalized YAML representation that you can
# version-control or hand off to a YAML-first workflow later.
from wellplot import render_report, render_window_png, save_report

report = demo.build_report(dataset)
report_yaml_path = WORKSPACE_RENDERS / "api_end_to_end_report.yaml"
save_report(report, report_yaml_path)
print("Saved report YAML:", report_yaml_path)

In [ ]:
# Render the full report to PDF and save one smaller PNG window for quick QC.
pdf_path = WORKSPACE_RENDERS / "api_end_to_end_demo.pdf"
png_path = WORKSPACE_RENDERS / "api_end_to_end_window.png"

pdf_result = render_report(report, output_path=pdf_path)
window_png = render_window_png(
    report,
    depth_range=(8360.0, 8420.0),
    depth_range_unit="ft",
    page_index=0,
    dpi=140,
)
png_path.write_bytes(window_png)

print("Pages:", pdf_result.page_count)
print("Saved PDF:", pdf_path)
print("Saved PNG:", png_path)

## Adapt End-to-End API Walkthrough Safely

- Keep the raw and processed datasets separate until the merge policy is settled.
- Save the normalized report YAML when you want a notebook-built layout to become a reusable file artifact.
- Use a small PNG window as a fast QC snapshot even if the final deliverable is PDF.
